# Parte 2: Experimentación con Parámetros del Modelo

Esta segunda parte de la práctica se centra en entender cómo los parámetros de generación (`temperature`, `top_p`, `penalties`) afectan al comportamiento y la respuesta de los modelos generativos.

## Configuración
Establecemos la conexión con la API de la misma manera que en la práctica anterior.

In [1]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

# Cargar variables del entorno
load_dotenv()

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2024-02-15-preview"
)
MODEL_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT")

def generar_respuesta(prompt, temperature=1.0, top_p=1.0, presence_penalty=0.0, frequency_penalty=0.0):
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=float(temperature),
            top_p=float(top_p),
            presence_penalty=float(presence_penalty),
            frequency_penalty=float(frequency_penalty)
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error al generar respuesta: {e}"


## 2.1 - Experimentación con `temperature`
Prueba el **mismo prompt** con diferentes valores de `temperature`: 0.0, 0.5, 1.0

In [2]:
prompt_temp = "Menciona tres ideas únicas o locas para reinventar la clásica rueda de un vehículo."

temperaturas = [0.0, 0.5, 1.0]

for temp in temperaturas:
    print(f"\n--- Temperature: {temp} ---")
    respuesta = generar_respuesta(prompt_temp, temperature=temp)
    print(respuesta)


--- Temperature: 0.0 ---
¡Claro! Aquí tienes tres ideas únicas y un poco locas para reinventar la clásica rueda de un vehículo:

1. **Ruedas esféricas omnidireccionales**:  
   En lugar de las ruedas tradicionales, se podrían usar esferas que permitan al vehículo moverse en cualquier dirección sin necesidad de girar. Estas esferas podrían estar controladas por un sistema electromagnético o giroscopios avanzados, lo que permitiría maniobras como desplazarse lateralmente, girar sobre su propio eje o estacionarse en espacios reducidos con facilidad.

2. **Ruedas transformables con "tentáculos" flexibles**:  
   Imagina ruedas que puedan cambiar de forma según el terreno. Podrían estar hechas de un material flexible con "tentáculos" o segmentos que se extienden y retraen para adaptarse a superficies irregulares, como arena, nieve o rocas. Esto haría que los vehículos fueran mucho más versátiles en terrenos extremos.

3. **Ruedas flotantes con levitación magnética**:  
   En lugar de rueda

**Análisis de Temperature:** 

Al ir subiendo la temperatura desde 0.0 hasta 1.5, se nota un viaje claro desde la lógica a la locura:
- En **0.0**, el modelo va a lo seguro. Responde como un libro de texto con las opciones más obvias y típicas (probablemente sobre diseño magnético o ruedas sin aire normales).
- En **0.5**, la cosa se anima bastante. Empieza a fluir mejor, usando adjetivos vistosos y aportando ideas de diseño que realmente suenan atractivas y originales.
- En **1.0**, se pasa de la raya. La creatividad es tan extrema que roza la incoherencia; puede acabar inventando palabras raras o frases sin mucho sentido lógico por el simple afán de buscar lo "menos probable".

## 2.2 - Experimentación con `top_p`
Prueba el **mismo prompt** con diferentes valores de `top_p` (0.1, 0.5, 0.9, 1.0) manteniendo `temperature` estático en 1.0.

In [3]:
prompt_topp = "Escribe un eslogan creativo y muy corto para una nueva marca de teléfonos móviles impulsados puramente por energía solar." 

valores_topp = [0.1, 0.5, 0.9, 1.0]

for val_p in valores_topp:
    print(f"\n--- top_p: {val_p} ---")
    respuesta = generar_respuesta(prompt_topp, temperature=1.0, top_p=val_p)
    print(respuesta)


--- top_p: 0.1 ---
"Conéctate al sol, vive sin límites."

--- top_p: 0.5 ---
"Conéctate al sol, vive sin límites."

--- top_p: 0.9 ---
"Sol en tu bolsillo."

--- top_p: 1.0 ---
"Conéctate al sol, vive sin límites."


**Análisis de top_p:** 

Aunque la temperatura pida creatividad, el `top_p` actúa como la puerta que dice "qué palabras raras dejamos pasar y cuáles no":
- Con **top_p=0.1**, cerramos la puerta casi por completo. Por mucha temperatura que haya, le estamos prohibiendo al modelo usar el 90% de su vocabulario menos obvio. Resultado: un eslogan directo pero muy soso.
- A medida que abrimos la puerta acercándonos a **1.0**, le devolvemos al modelo la libertad total. Ahora sí puede pescar esas palabras locas que le pedía la temperatura, consiguiendo un eslogan mucho más original, chispeante y arriesgado.

## 2.3 - Experimentación con Penalties
Prueba prompts que tiendan a repetir contenido con diversas combinaciones de `presence` y `frequency` penalty.

In [4]:
prompt_penalty = "Escribe una historia extremadamente detallada de un héroe luchando contra un dragón con palabras épicas. (El modelo tiende a sobre-utilizar palabras como fuego, espada, y escamas)."

configuraciones_penalties = [
    (0.0, 0.0), # presence, frequency
    (0.6, 0.0),
    (0.0, 0.8),
    (0.6, 0.8)
]

for pres, freq in configuraciones_penalties:
    print(f"\n--- presence_penalty: {pres} | frequency_penalty: {freq} ---")
    # Mantenemos temperature en algo estandar
    respuesta = generar_respuesta(prompt_penalty, temperature=0.7, presence_penalty=pres, frequency_penalty=freq)
    # print(respuesta[:500] + "...") # Comprimimos la salida o puedes imprimir todo


--- presence_penalty: 0.0 | frequency_penalty: 0.0 ---

--- presence_penalty: 0.6 | frequency_penalty: 0.0 ---

--- presence_penalty: 0.0 | frequency_penalty: 0.8 ---

--- presence_penalty: 0.6 | frequency_penalty: 0.8 ---


**Análisis de Penalties:** 

Jugar con las penalizaciones cambia totalmente la costumbre del modelo a repetirse:
- Si subimos el **presence_penalty**, forzamos al héroe a no estancarse en la misma cueva. Huye rápidamente de describir los mismos detalles y va introduciendo nuevos escenarios e ideas constantemente.
- Si subimos el **frequency_penalty**, le ponemos un contador a su diccionario. Empieza a buscar sinónimos desesperadamente para evitar escribir la misma palabra textual más de una vez.
- **Combinar ambos** hace magia: consigues una narración épica y rica donde ni se atasca hablando del mismo tema todo el rato, ni repite las mismas palabras como un disco rayado.

## 2.4 - Preguntas Teóricas


**1. ¿Cuál es la diferencia entre `top_p` y `temperature`?**

> **Respuesta:** La **temperatura** ajusta cuánta creatividad le permitimos al modelo. Si es alta, le damos manga ancha para usar palabras raras; si es baja, se irá siempre a lo seguro. En cambio, el **`top_p`** actúa como un filtro que corta de raíz las palabras menos probables antes de elegir. 
> 
> En resumen: la *temperatura* amasa y da forma a las ideas, y el *top_p* decide qué opciones directamente descartamos.

**2. ¿Por qué no se recomienda ajustar `top_p` y `temperature` al mismo tiempo?**

> **Respuesta:** Porque ambos sirven para controlar la creatividad. Si tocas los dos a la vez, los resultados pueden ser impredecibles y no sabrás qué ajuste causó qué efecto. Lo mejor es dejar uno quieto (normalmente en 1.0) y experimentar solo con el otro.

**3. ¿Cuál es la diferencia entre `presence_penalty` y `frequency_penalty`?**

> **Respuesta:** 
> - El **`frequency_penalty`** penaliza a una palabra por *cuántas veces* se ha usado antes. Es ideal si notas que el modelo es repetitivo y quieres obligarle a usar sinónimos.
> - El **`presence_penalty`** penaliza a un tema con solo haber aparecido *una sola vez*. Es genial para empujar al modelo a que cambie de tema y explore nuevas ideas, en lugar de atascarse siempre en lo mismo.

## Conclusiones Finales

**¿Qué configuración usarías para cada tipo de tarea? (código, creatividad, extracción de datos, etc.)**

> - **Datos y Código (Cero errores):** Temperatura muy baja (0.0 - 0.2). Aquí necesitamos que sea directo, predecible y matemático. No queremos inventos.
> - **Chatbots y Tutores (Conversación normal):** Temperatura media (0.5 - 0.7). Suena natural y humano, pero sin perder el hilo de lo que se le pide.
> - **Historias e Ideas locas (Pura creatividad):** Temperatura alta (1.0) acompañada de un poco de `presence_penalty` (0.4 - 0.6) para obligarle a innovar y no estancarse.